# SDXL LoRA Generation

Generates paired Run A (clean) and Run B (cloaked) images using an
identical prompt, seed sequence, sampler settings, and resolution -- only
the LoRA weights differ between runs, matching the thesis's downstream
generative validation methodology.

### Before running
Attach a Kaggle Dataset containing:
- `lora_run_a.safetensors`
- `lora_run_b.safetensors`
- optional: `sd_xl_base_1.0.safetensors`, to load SDXL locally without
  needing internet access for the base model download

**Settings:** enable GPU (T4/P100) and, if you don't have a local SDXL
file, enable Internet access.

## Step 0 — Environment check

In [ ]:
import sys
from pathlib import Path
import glob

print("Python:", sys.version)
print("\nKaggle input folders:")
for p in glob.glob("/kaggle/input/*"):
    print(" -", p)

## Step 1 — Install dependencies

In [ ]:
import subprocess, sys

packages = [
    "diffusers>=0.27.0",
    "transformers",
    "accelerate",
    "safetensors",
    "invisible_watermark",
    "sentencepiece",
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)

import diffusers
print(f"Installed diffusers version: {diffusers.__version__}")
print("(record this version when reporting results -- >=0.27.0 is not pinned to an exact build)")

## Step 2 — Locate model files


In [ ]:
from pathlib import Path

SEARCH_ROOT = Path("/kaggle/input")

def find_file(filename: str, root: Path = SEARCH_ROOT) -> Path | None:
    matches = list(root.rglob(filename))
    return matches[0] if matches else None

LORA_A = find_file("lora_run_a.safetensors")
LORA_B = find_file("lora_run_b.safetensors")
SDXL_LOCAL = find_file("sd_xl_base_1.0.safetensors")

print("Run A LoRA :", LORA_A or "NOT FOUND")
print("Run B LoRA :", LORA_B or "NOT FOUND")
print("Local SDXL :", SDXL_LOCAL or "not found -- will load from Hugging Face instead")

missing = [name for name, path in [("lora_run_a.safetensors", LORA_A), ("lora_run_b.safetensors", LORA_B)] if path is None]
if missing:
    raise FileNotFoundError(
        f"Required file(s) not found under {SEARCH_ROOT}: {missing}. "
        "Check that your Kaggle Dataset is attached via 'Add Input'."
    )

## Step 3 — Generation configuration

In [ ]:
from pathlib import Path

PROMPT = "a photo of ohwx person"

NUM_IMAGES = 100      # 50 minimum, 100 stronger
BASE_SEED = 42
HEIGHT = 512
WIDTH = 512
STEPS = 25
GUIDANCE_SCALE = 7.5

OUT_ROOT = Path("/kaggle/working/generated")
RUN_A_DIR = OUT_ROOT / "run_a"
RUN_B_DIR = OUT_ROOT / "run_b"

RUN_A_DIR.mkdir(parents=True, exist_ok=True)
RUN_B_DIR.mkdir(parents=True, exist_ok=True)

print("Prompt:", PROMPT)
print("Images per run:", NUM_IMAGES)

## Step 4 — Load SDXL pipeline


In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

if SDXL_LOCAL is not None:
    print("Loading local SDXL safetensors:", SDXL_LOCAL)
    pipe = StableDiffusionXLPipeline.from_single_file(
        str(SDXL_LOCAL),
        torch_dtype=torch.float16,
        use_safetensors=True,
    )
else:
    print("Local SDXL file not found. Loading from Hugging Face...")
    pipe = StableDiffusionXLPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True,
    )

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to("cuda")
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()

print("Pipeline loaded")

## Step 5 — Generation helper

In [ ]:
import gc
from pathlib import Path
from tqdm.auto import tqdm


def generate_run(run_label: str, lora_path: Path, output_dir: Path) -> None:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    try:
        pipe.unload_lora_weights()
    except Exception:
        pass

    print(f"\nLoading {run_label} LoRA: {lora_path}")
    pipe.load_lora_weights(str(Path(lora_path).parent), weight_name=Path(lora_path).name)

    print(f"Generating {NUM_IMAGES} images for {run_label}...")
    for i in tqdm(range(NUM_IMAGES)):
        seed = BASE_SEED + i
        generator = torch.Generator(device="cuda").manual_seed(seed)

        image = pipe(
            prompt=PROMPT,
            num_inference_steps=STEPS,
            guidance_scale=GUIDANCE_SCALE,
            height=HEIGHT,
            width=WIDTH,
            generator=generator,
        ).images[0]

        image.save(output_dir / f"gen_{i:03d}.png")

    print(f"{run_label} complete:", len(list(output_dir.glob("*.png"))), "images")

    gc.collect()
    torch.cuda.empty_cache()

## Step 6 — Generate Run A (clean LoRA)

In [ ]:
generate_run(
    run_label="Run A Clean-LoRA",
    lora_path=LORA_A,
    output_dir=RUN_A_DIR,
)

## Step 7 — Generate Run B (cloaked LoRA)

Uses the exact same prompt and seed sequence as Run A.

In [ ]:
generate_run(
    run_label="Run B Cloaked-LoRA",
    lora_path=LORA_B,
    output_dir=RUN_B_DIR,
)

## Step 8 — Verify paired outputs

In [ ]:
run_a_files = sorted(p.name for p in RUN_A_DIR.glob("*.png"))
run_b_files = sorted(p.name for p in RUN_B_DIR.glob("*.png"))

print("Run A images:", len(run_a_files))
print("Run B images:", len(run_b_files))
print("Filenames match:", run_a_files == run_b_files)
print("First files:", run_a_files[:5])

## Step 9 — Zip outputs for download

In [ ]:
import shutil

zip_base = "/kaggle/working/generated_lora_runs"
zip_path = shutil.make_archive(zip_base, "zip", OUT_ROOT)
print("ZIP saved at:", zip_path)